# Lean NMI Experiment (CPU, ~15 min)

Runs 0.5B model, 5 seeds x 2 tasks with DPO, pruning, circuit analysis.
No GPU needed. Manual LoRA (no peft dependency).

In [ ]:
# Force CPU before any torch import
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

import torch
torch.cuda.is_available = lambda: False
torch.cuda.device_count = lambda: 0

# Write the experiment script to a file and exec it
import base64, sys
b64 = (
    "..."
)
script = __import__("base64").b64decode("".join(b64)).decode()
print(f'Running lean NMI experiment ({len(script)} bytes)...')
exec(script)


In [ ]:
import zipfile, os, json
if os.path.exists('nmi_results'):
    files = sorted(os.listdir('nmi_results'))
    with zipfile.ZipFile('nmi_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for f in files:
            fp = os.path.join('nmi_results', f)
            if os.path.isfile(fp):
                z.write(fp)
    print(f'Packaged {len(files)} files')
    for f in files:
        if f.endswith(".json"):
            d = json.load(open(os.path.join("nmi_results", f)))
            b = d.get("baseline", {})
            dp = d.get("dpo", {}).get("after", {})
            print(f"  {f}: ASR={b.get('asr',0):.3f} benign={b.get('benign_acc',0):.3f} DPO={dp.get('asr',0):.3f}")
    print('\nDownload nmi_results.zip from Output')
